In [1]:
# %pip install -q langchain-openai langchain-core requests

In [2]:
# ============================================================================
# LangChain Tool Usage Tutorial
# Demonstrates tool creation, binding, and agent execution with Cohere LLM
# ============================================================================

# ----------------------------------------------------------------------------
# 1. ENVIRONMENT SETUP
# --------------------

In [3]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve and validate Cohere API key
api_token = os.getenv("COHERE_API_KEY")
if not api_token:
    raise ValueError("COHERE_API_KEY not found in .env file")


In [4]:
# ----------------------------------------------------------------------------
# 2. LLM INITIALIZATION
# ----------------------------------------------------------------------------

In [5]:
from langchain_cohere import ChatCohere
from langchain_core.messages import AIMessage, HumanMessage

# Initialize Cohere language model
llm = ChatCohere(
    cohere_api_key=api_token,
    model="command-a-03-2025"
)

# Test basic invocation (optional)
# result = llm.invoke("What is the capital of France?")
# print(result)


c:\Biprayan\GenAI-LangChain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# ----------------------------------------------------------------------------
# 3. TOOL CREATION - SIMPLE MULTIPLICATION TOOL
# ----------------------------------------------------------------------------

In [7]:
from langchain_core.tools import tool
import requests

@tool
def multiply(a: int, b: int) -> int:
    """
    Multiplies two integers and returns their product.
    
    Args:
        a (int): First number
        b (int): Second number
    
    Returns:
        int: Product of a and b
    """
    return a * b


In [8]:
# Test the tool directly
print("Direct tool invocation test:")
print(multiply.invoke({'a': 3, 'b': 4}))  # Output: 12

Direct tool invocation test:
12


In [9]:
# Inspect tool properties
print(f"\nTool Name: {multiply.name}")
print(f"Tool Description: {multiply.description}")
print(f"Tool Arguments: {multiply.args}")



Tool Name: multiply
Tool Description: Multiplies two integers and returns their product.

Args:
    a (int): First number
    b (int): Second number

Returns:
    int: Product of a and b
Tool Arguments: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [10]:
# ----------------------------------------------------------------------------
# 4. TOOL BINDING - CONNECTING TOOLS TO LLM
# ----------------------------------------------------------------------------

In [11]:
# Create LLM instance with bound tools
llm_with_tools = llm.bind_tools([multiply])

# Test without tool usage (regular chat)
print("\n--- Testing regular chat (no tool needed) ---")
response = llm_with_tools.invoke('Hi how are you')
print(response.content)


--- Testing regular chat (no tool needed) ---
Hi there! As an AI language model, I don't have feelings or emotions, but I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?


In [12]:
# ----------------------------------------------------------------------------
# 5. TOOL EXECUTION WORKFLOW - MANUAL APPROACH
# ----------------------------------------------------------------------------

In [13]:
# Create user query that requires tool usage
query = HumanMessage('can you multiply 3 with 1000')
messages = [query]

print("\n--- Step 1: Initial messages ---")
print(messages)


--- Step 1: Initial messages ---
[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]


In [14]:
# Invoke LLM with tool-bound instance
result = llm_with_tools.invoke(messages)
print("\n--- Step 2: LLM response with tool call ---")
print(result)


--- Step 2: LLM response with tool call ---
content='I will use the multiply tool to multiply 3 with 1000.' additional_kwargs={'id': '07543f2f-0906-4401-addd-cdfc3cb427ac', 'finish_reason': 'TOOL_CALL', 'tool_plan': 'I will use the multiply tool to multiply 3 with 1000.', 'tool_calls': [{'id': 'multiply_5p6f16dh0xc8', 'type': 'function', 'function': {'name': 'multiply', 'arguments': '{"a":3,"b":1000}'}}], 'token_count': {'input_tokens': 1490.0, 'output_tokens': 61.0}} response_metadata={'id': '07543f2f-0906-4401-addd-cdfc3cb427ac', 'finish_reason': 'TOOL_CALL', 'tool_plan': 'I will use the multiply tool to multiply 3 with 1000.', 'tool_calls': [{'id': 'multiply_5p6f16dh0xc8', 'type': 'function', 'function': {'name': 'multiply', 'arguments': '{"a":3,"b":1000}'}}], 'token_count': {'input_tokens': 1490.0, 'output_tokens': 61.0}} id='run--58d565e8-f455-4b77-83fd-62bc25ce8c6a-0' tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'multiply_5p6f16dh0xc8', 'type': 'tool_call'

In [15]:
# Append AI response to message history
messages.append(result)
print("\n--- Step 3: Messages after AI response ---")
print(messages)


--- Step 3: Messages after AI response ---
[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}), AIMessage(content='I will use the multiply tool to multiply 3 with 1000.', additional_kwargs={'id': '07543f2f-0906-4401-addd-cdfc3cb427ac', 'finish_reason': 'TOOL_CALL', 'tool_plan': 'I will use the multiply tool to multiply 3 with 1000.', 'tool_calls': [{'id': 'multiply_5p6f16dh0xc8', 'type': 'function', 'function': {'name': 'multiply', 'arguments': '{"a":3,"b":1000}'}}], 'token_count': {'input_tokens': 1490.0, 'output_tokens': 61.0}}, response_metadata={'id': '07543f2f-0906-4401-addd-cdfc3cb427ac', 'finish_reason': 'TOOL_CALL', 'tool_plan': 'I will use the multiply tool to multiply 3 with 1000.', 'tool_calls': [{'id': 'multiply_5p6f16dh0xc8', 'type': 'function', 'function': {'name': 'multiply', 'arguments': '{"a":3,"b":1000}'}}], 'token_count': {'input_tokens': 1490.0, 'output_tokens': 61.0}}, id='run--58d565e8-f455-4b77-83fd-62bc25ce8c6a-0', 

In [16]:
# Execute the tool with the arguments provided by the LLM
tool_result = multiply.invoke(result.tool_calls[0])
print("\n--- Step 4: Tool execution result ---")
print(tool_result)


--- Step 4: Tool execution result ---
content='3000' name='multiply' tool_call_id='multiply_5p6f16dh0xc8'


In [17]:
# Append tool result to message history
messages.append(tool_result)
print("\n--- Step 5: Complete message chain ---")
print(messages)


--- Step 5: Complete message chain ---
[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}), AIMessage(content='I will use the multiply tool to multiply 3 with 1000.', additional_kwargs={'id': '07543f2f-0906-4401-addd-cdfc3cb427ac', 'finish_reason': 'TOOL_CALL', 'tool_plan': 'I will use the multiply tool to multiply 3 with 1000.', 'tool_calls': [{'id': 'multiply_5p6f16dh0xc8', 'type': 'function', 'function': {'name': 'multiply', 'arguments': '{"a":3,"b":1000}'}}], 'token_count': {'input_tokens': 1490.0, 'output_tokens': 61.0}}, response_metadata={'id': '07543f2f-0906-4401-addd-cdfc3cb427ac', 'finish_reason': 'TOOL_CALL', 'tool_plan': 'I will use the multiply tool to multiply 3 with 1000.', 'tool_calls': [{'id': 'multiply_5p6f16dh0xc8', 'type': 'function', 'function': {'name': 'multiply', 'arguments': '{"a":3,"b":1000}'}}], 'token_count': {'input_tokens': 1490.0, 'output_tokens': 61.0}}, id='run--58d565e8-f455-4b77-83fd-62bc25ce8c6a-0', tool

In [18]:
# Get final answer from LLM using tool result
final_response = llm_with_tools.invoke(messages).content
print("\n--- Step 6: Final answer ---")
print(final_response)


--- Step 6: Final answer ---
The product of 3 and 1000 is **3000**.
